In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd
from show import *

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [5]:
files_fdt = sorted(glob.glob('/home/ulyanov/data/stereo/2024-03-30/fdt/*'))
files_fdt

['/home/ulyanov/data/stereo/2024-03-30/fdt/solo_L2_phi-fdt-bamb_20240330T044009_V202602220014_0443301531.fits.gz',
 '/home/ulyanov/data/stereo/2024-03-30/fdt/solo_L2_phi-fdt-bazi_20240330T044009_V202602220014_0443301531.fits.gz',
 '/home/ulyanov/data/stereo/2024-03-30/fdt/solo_L2_phi-fdt-binc_20240330T044009_V202602220014_0443301531.fits.gz',
 '/home/ulyanov/data/stereo/2024-03-30/fdt/solo_L2_phi-fdt-blos_20240330T044009_V202602220014_0443301531.fits.gz']

In [6]:
file_fdt = files_fdt[3]

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data


did = int(file_fdt.split('_')[-1].split('.')[0])
data_fdt = undistort(data_fdt, header_fdt, xd, yd) * 1.1

view_fdt = View.from_header(header_fdt).update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                                               crota=header_fdt['CROTA'] + 0.25)
view_new = view_fdt.update(nx=1024, ny=1024, rsun=480, xc=511.5, yc=511.5, crota=0, rsun_arc=0)

data_fdt = view_new.reproject(data_fdt, view_fdt)
show_data(data_fdt, view_new, 'FDT', label=r'$B_{los}, G$', vmin=-1000, vmax=1000)

In [9]:
files_hmi = sorted(glob.glob('/home/ulyanov/data/stereo/2024-03-30/hmi/*'))
files_hmi

['/home/ulyanov/data/stereo/2024-03-30/hmi/hmi.M_45s.20240330_044545_TAI.2.magnetogram.fits']

In [10]:
file_hmi = files_hmi[0]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data


data_hmi = rebin(data_hmi, 6, update_header=header_hmi)
view_hmi = View.from_header(header_hmi)

#view_new = view_hmi.update(nx=1024, ny=1024, rsun=480, xc=511.5, yc=511.5, crota=0, rsun_arc=0)
data_hmi = view_new.reproject(data_hmi, view_hmi, mu_thr=0.1)

show_data(data_hmi, view_new, 'HMI', label=r'$B_{los}, G$', vmin=-1000, vmax=1000)

In [7]:
transform = view_hmi.to_carrington(origin='heliographic') - view_fdt.to_carrington(origin='heliographic')

e1 = (0,0,1)
e2 = transform(e1)[0]

e1 = np.array(e1)
e2 = np.array(e2)

q = np.sum(e1 * e2)
delta = np.sqrt(1 - q ** 2)

e2_ = (e2 - e1 * q) / delta

v1 = data_fdt.copy()
v2 = data_hmi.copy()

v2[np.abs(v2) < 15] = np.nan

v2_ = (v2 - v1 * q) / delta

print(e2_, np.arccos(q) * 180 / np.pi)

[0.9992635  0.03837274 0.        ] 37.10881305610327


In [8]:
with fits.open(files_fdt[1]) as hdul:
    header_azi = hdul[0].header
    data_azi = hdul[0].data

data_azi = view_new.reproject(data_azi, view_fdt)

crota = header_azi['CROTA'] + 0.25
data_azi += crota

In [9]:
amb = ((np.sin(data_azi * np.pi / 180) * v2_) > 0)

azi = (data_azi + amb * 180) % 360
azi[np.isnan(v2)] = np.nan

show_data(azi, view_new, title='Stereoscopy', label='Azimuth, degrees', cmap='hsv', vmin=0, vmax=360)
plt.xlim(840,1040)
plt.ylim(300,500)

(300.0, 500.0)

In [10]:
with fits.open(files_fdt[0]) as hdul:
    header_amb = hdul[0].header
    data_amb = hdul[0].data[0]

data_amb = view_new.reproject(data_amb, view_fdt)

In [11]:
azi = (data_azi + (data_amb > 4) * 180) % 360
azi[np.isnan(v2)] = np.nan

show_data(azi, view_new, title='Minimum energy', label='Azimuth, degrees', cmap='hsv', vmin=0, vmax=360)
plt.xlim(840,1040)
plt.ylim(300,500)

(300.0, 500.0)